# Qwen QLoRA checkpoint-3000 eval-only

The notebook loads a saved PEFT/LoRA checkpoint, evaluates it on validation/test JSONL files, selects the threshold on validation, and saves metrics and predictions to `/kaggle/working`.

Default configuration targets the ordinary QLoRA checkpoint trained on `instruction_temporal`. For ReLLa-lite prompts, set `DATA_KIND = "retrieved"`.


In [ ]:
!pip -q install -U "transformers>=4.51.0" "peft>=0.14.0" "accelerate>=1.2.0" bitsandbytes scikit-learn pandas tqdm safetensors

In [ ]:
import os, json, math, shutil, zipfile, glob, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# 1. Configure run
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
DATA_KIND = "ordinary"  # "ordinary" for instruction_temporal, "retrieved" for instruction_temporal_retrieved_topk
EVAL_PER_CLASS = 1000   # 1000 -> 2000 val and 2000 test, set 500 for faster check
MAX_LENGTH = 1024
BATCH_SIZE = 4
OUT_DIR = Path("/kaggle/working/checkpoint3000_eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Locate adapter checkpoint
def unzip_checkpoint_if_needed():
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        if "checkpoint" in os.path.basename(z).lower() or "adapter" in os.path.basename(z).lower() or "qlora" in os.path.basename(z).lower():
            dst = Path("/kaggle/working/unzipped_checkpoint")
            dst.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z, "r") as f:
                f.extractall(dst)
            print("Unzipped:", z, "->", dst)
            return
unzip_checkpoint_if_needed()
adapter_candidates = []
for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files and "adapter_model.safetensors" in files:
        adapter_candidates.append(root)
for root, dirs, files in os.walk("/kaggle/working/unzipped_checkpoint"):
    if "adapter_config.json" in files and "adapter_model.safetensors" in files:
        adapter_candidates.append(root)
print("Adapter candidates:")
for i, p in enumerate(adapter_candidates):
    print(i, p)
assert adapter_candidates, "No PEFT adapter checkpoint found. Check that checkpoint-3000 was added as Kaggle input."
ADAPTER_PATH = adapter_candidates[0]
print("Using ADAPTER_PATH =", ADAPTER_PATH)

In [ ]:
# 3. Locate JSONL data
jsonl_dirs = []
for root, dirs, files in os.walk("/kaggle/input"):
    if {"train.jsonl", "val.jsonl", "test.jsonl"}.issubset(set(files)):
        jsonl_dirs.append(root)
print("JSONL candidates:")
for i, p in enumerate(jsonl_dirs):
    print(i, p)
assert jsonl_dirs, "No train/val/test JSONL directory found. Add instruction_temporal dataset as Kaggle input."
if DATA_KIND == "retrieved":
    preferred = [p for p in jsonl_dirs if "retrieved" in p.lower() or "topk" in p.lower() or "rella" in p.lower()]
else:
    preferred = [p for p in jsonl_dirs if "retrieved" not in p.lower() and "topk" not in p.lower() and "rella" not in p.lower()]
DATA_DIR = preferred[0] if preferred else jsonl_dirs[0]
print("Using DATA_DIR =", DATA_DIR)

In [ ]:
# 4. Load validation/test and create balanced eval subsets
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def normalize_label(x):
    if isinstance(x, str):
        return 1 if x.strip().lower().startswith("yes") else 0
    return int(x)

val_df = read_jsonl(os.path.join(DATA_DIR, "val.jsonl"))
test_df = read_jsonl(os.path.join(DATA_DIR, "test.jsonl"))
for df in [val_df, test_df]:
    if "output" in df.columns:
        df["label"] = df["output"].map(normalize_label)
    elif "label" in df.columns:
        df["label"] = df["label"].map(normalize_label)
    else:
        raise ValueError("Expected output or label column in JSONL data.")

def balanced_sample(df, per_class=1000, seed=42):
    parts = []
    for y in [0, 1]:
        sub = df[df["label"] == y]
        n = min(per_class, len(sub))
        parts.append(sub.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

val_eval = balanced_sample(val_df, EVAL_PER_CLASS, SEED)
test_eval = balanced_sample(test_df, EVAL_PER_CLASS, SEED)
print("val_eval:", val_eval.shape, val_eval["label"].value_counts().to_dict())
print("test_eval:", test_eval.shape, test_eval["label"].value_counts().to_dict())
print("Columns:", list(val_eval.columns))
display(val_eval.head(2))

In [ ]:
# 5. Prompt builder
def row_to_user_text(row):
    if "messages" in row and isinstance(row["messages"], list):
        msgs = [m for m in row["messages"] if m.get("role") == "user"]
        return "\n".join([m.get("content", "") for m in msgs]).strip()
    instruction = str(row.get("instruction", "")).strip()
    inp = str(row.get("input", "")).strip()
    if instruction and inp:
        return instruction + "\n\n" + inp
    if instruction:
        return instruction
    if inp:
        return inp
    if "prompt" in row:
        return str(row["prompt"]).strip()
    raise ValueError("Cannot build prompt. Expected instruction/input/messages/prompt columns.")

def build_prompt(row):
    user_text = row_to_user_text(row)
    messages = [{"role": "user", "content": user_text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
# 6. Load tokenizer, base model and adapter
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Loaded adapter:", ADAPTER_PATH)
print("compute_dtype:", compute_dtype)

In [ ]:
# 7. Yes/No probability scoring
yes_ids = tokenizer.encode("Yes", add_special_tokens=False)
no_ids = tokenizer.encode("No", add_special_tokens=False)
yes_space_ids = tokenizer.encode(" Yes", add_special_tokens=False)
no_space_ids = tokenizer.encode(" No", add_special_tokens=False)
print("Yes ids:", yes_ids, "No ids:", no_ids)
print("' Yes' ids:", yes_space_ids, "' No' ids:", no_space_ids)

def first_token_id(ids):
    assert len(ids) >= 1
    return ids[0]

YES_ID = first_token_id(yes_ids if len(yes_ids) == 1 else yes_space_ids)
NO_ID = first_token_id(no_ids if len(no_ids) == 1 else no_space_ids)
print("Using YES_ID =", YES_ID, "NO_ID =", NO_ID)

@torch.inference_mode()
def score_yes_probability(df):
    probs = []
    prompts = [build_prompt(row) for _, row in df.iterrows()]
    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        enc = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(model.device)
        out = model(**enc)
        lengths = enc["attention_mask"].sum(dim=1) - 1
        logits = out.logits[torch.arange(len(batch_prompts), device=model.device), lengths, :]
        pair_logits = torch.stack([logits[:, NO_ID], logits[:, YES_ID]], dim=1)
        batch_probs = torch.softmax(pair_logits, dim=1)[:, 1].detach().float().cpu().numpy()
        probs.extend(batch_probs.tolist())
        del enc, out, logits, pair_logits
        torch.cuda.empty_cache()
    return np.array(probs)

In [ ]:
# 8. Run evaluation
val_probs = score_yes_probability(val_eval)
test_probs = score_yes_probability(test_eval)
val_eval = val_eval.copy()
test_eval = test_eval.copy()
val_eval["prob_yes"] = val_probs
test_eval["prob_yes"] = test_probs
val_eval[["label", "prob_yes"]].head()

In [ ]:
# 9. Select threshold on validation and compute metrics
def metrics_at_threshold(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, prob)),
        "pr_auc": float(average_precision_score(y_true, prob)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "threshold": float(threshold)
    }

threshold_grid = np.linspace(0.05, 0.95, 181)
val_y = val_eval["label"].values
test_y = test_eval["label"].values
best_threshold = max(threshold_grid, key=lambda t: f1_score(val_y, (val_probs >= t).astype(int), zero_division=0))
val_metrics = metrics_at_threshold(val_y, val_probs, best_threshold)
test_metrics = metrics_at_threshold(test_y, test_probs, best_threshold)
metrics = {
    "base_model": BASE_MODEL,
    "adapter_path": ADAPTER_PATH,
    "data_dir": DATA_DIR,
    "data_kind": DATA_KIND,
    "checkpoint": "checkpoint-3000",
    "eval_per_class": EVAL_PER_CLASS,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "validation": val_metrics,
    "test": test_metrics
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))

In [ ]:
# 10. Save artifacts
val_out = val_eval.copy()
test_out = test_eval.copy()
keep_cols = [c for c in ["instruction", "input", "output", "label", "prob_yes"] if c in val_out.columns]
val_out[keep_cols].to_csv(OUT_DIR / "val_predictions_checkpoint3000.csv", index=False)
test_out[keep_cols].to_csv(OUT_DIR / "test_predictions_checkpoint3000.csv", index=False)
with open(OUT_DIR / "metrics_checkpoint3000.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
summary = f'''# Qwen QLoRA checkpoint-3000 evaluation

Base model: `{BASE_MODEL}`  
Adapter: `{ADAPTER_PATH}`  
Data: `{DATA_DIR}`  
Eval subset: `{EVAL_PER_CLASS}` examples per class  
Threshold selected on validation: `{best_threshold:.4f}`

## Validation

- ROC-AUC: `{val_metrics["roc_auc"]:.6f}`
- PR-AUC: `{val_metrics["pr_auc"]:.6f}`
- Accuracy: `{val_metrics["accuracy"]:.6f}`
- Precision: `{val_metrics["precision"]:.6f}`
- Recall: `{val_metrics["recall"]:.6f}`
- F1: `{val_metrics["f1"]:.6f}`

## Test

- ROC-AUC: `{test_metrics["roc_auc"]:.6f}`
- PR-AUC: `{test_metrics["pr_auc"]:.6f}`
- Accuracy: `{test_metrics["accuracy"]:.6f}`
- Precision: `{test_metrics["precision"]:.6f}`
- Recall: `{test_metrics["recall"]:.6f}`
- F1: `{test_metrics["f1"]:.6f}`
'''
with open(OUT_DIR / "qwen_qlora_checkpoint3000_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)
zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("Saved artifacts to:", OUT_DIR)
print("Zip:", zip_path)
print(os.listdir(OUT_DIR))

In [ ]:
# 11. Optional: inspect a few predictions
cols = [c for c in ["output", "label", "prob_yes", "instruction", "input"] if c in test_eval.columns]
display(test_eval.sort_values("prob_yes", ascending=False)[cols].head(5))
display(test_eval.sort_values("prob_yes", ascending=True)[cols].head(5))